In [0]:
%sql

USE CATALOG medalhao;
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (StringType, IntegerType, DoubleType, DecimalType, DateType, TimestampType)

catalogo = "medalhao"
bronze_db = "bronze"
silver_db = "silver"

Deduplicação Sênior

In [0]:
# Remove duplicidades mantendo a versão mais recente do registro
def deduplicar_por_ingestao(df, coluna_id, coluna_ingestao="ingestion_datetime"):
    janela = Window.partitionBy(coluna_id).orderBy(F.col(coluna_ingestao).desc())
    
    return (
        df.select("*", F.row_number().over(janela).alias("rn"))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

Processamento de Tabela: Cotação do Dólar

In [0]:
# Leitura da tabela Bronze
df_cotacao_raw = spark.table(f"{catalogo}.{bronze_db}.tb_cotacao_dolar")

# Seleção, tipagem e extração da data
df_cotacao_tratada = df_cotacao_raw.select(
    F.col("cotacaoCompra").cast(DecimalType(10, 4)).alias("cotacao_dolar_compra"),
    F.to_timestamp("dataHoraCotacao").alias("data_hora_cotacao"),
    F.to_date("dataHoraCotacao").alias("data_referencia"),
    F.col("ingestion_datetime")
).filter(F.col("data_referencia").isNotNull())

# Deduplicação diária
janela_dia = Window.partitionBy("data_referencia").orderBy(F.col("ingestion_datetime").desc())
df_cotacao_por_dia = (
    df_cotacao_tratada
    .select("*", F.row_number().over(janela_dia).alias("rn"))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

limites_datas = df_cotacao_por_dia.select(
    F.min("data_referencia").alias("data_min"),
    F.max("data_referencia").alias("data_max")
).collect()[0]

data_min = limites_datas["data_min"]
data_max = limites_datas["data_max"]

# Calendário diário cobrindo fins de semana e feriados
df_calendario = spark.sql(f"""
    SELECT explode(sequence(to_date('{data_min}'), to_date('{data_max}'), interval 1 day)) AS data_referencia
""")

# Cruzar calendário com as cotações reais
df_cotacao_calendario = df_calendario.join(df_cotacao_por_dia, on="data_referencia", how="left")

# Preenchimento de lacunas
janela_ffill = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, 0)

df_dim_cotacao_dolar = (
    df_cotacao_calendario
    .withColumns({
        "cotacao_dolar_compra": F.last("cotacao_dolar_compra", ignorenulls=True).over(janela_ffill),
        "data_hora_cotacao": F.last("data_hora_cotacao", ignorenulls=True).over(janela_ffill),
        "ingestion_datetime": F.last("ingestion_datetime", ignorenulls=True).over(janela_ffill)
    })
    .select(
        F.col("data_referencia").cast(DateType()).alias("data_referencia"),
        F.round(F.col("cotacao_dolar_compra"), 4).alias("cotacao_dolar_compra"),
        F.col("data_hora_cotacao").cast(TimestampType()).alias("data_hora_cotacao"),
        F.col("ingestion_datetime").cast(TimestampType()).alias("ingestion_datetime")
    )
)

# Persistência da Silver em Delta
(
    df_dim_cotacao_dolar.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_cotacao_dolar")
)

print("Tabela silver.tb_cotacao_dolar processada com SUCESSO!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tabela silver.tb_cotacao_dolar processada com SUCESSO!


Processamento de Tabela: Informações dos Filmes

In [0]:
df_info_raw = spark.table(f"{catalogo}.{bronze_db}.tb_movies_info")

# Regra de Normalização e Tradução do Status
status_normalizado = F.upper(F.trim(F.regexp_replace(F.col("status"), r"[-_]", " ")))

status_traduzido = (
    F.when(status_normalizado == "RELEASED", "Lançado")
     .when(status_normalizado == "POST PRODUCTION", "Pós-Produção")
     .when(status_normalizado == "IN PRODUCTION", "Em Produção")
     .when(status_normalizado == "PLANNED", "Planejado")
     .when(status_normalizado == "RUMORED", "Rumores")
     .when(status_normalizado.isin("CANCELED", "CANCELLED"), "Cancelado")
     .otherwise("Não Informado")
)

# Tratamento de Data
release_date_clean = F.trim(F.col("release_date"))

data_lancamento_convertida = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
    F.expr("try_to_date(release_date, 'yyyy/MM/dd')"),
    F.expr("try_to_date(release_date, 'yyyy-dd-MM')"),
    F.expr("try_to_date(release_date, 'MM/dd/yyyy')")
)

# Projeção, Mapeamento em Português, Tipagem e Coluna Derivada
df_info_cleaned = (
    df_info_raw
    .select(
        F.col("id").cast(StringType()).alias("id_filme"),
        F.trim(F.col("title")).cast(StringType()).alias("titulo"),
        F.trim(F.col("original_title")).cast(StringType()).alias("titulo_original"),
        data_lancamento_convertida.alias("data_lancamento"),
        F.year(data_lancamento_convertida).alias("ano_lancamento"),
        F.expr("try_cast(runtime as int)").alias("duracao_minutos"),
        F.trim(F.col("original_language")).cast(StringType()).alias("idioma_original"),
        status_traduzido.alias("status_filme"),
        F.trim(F.col("overview")).cast(StringType()).alias("sinopse"),
        F.trim(F.col("tagline")).cast(StringType()).alias("frase_divulgacao"),
        F.col("ingestion_datetime")
    )
    .filter(F.col("id_filme").isNotNull())
)

# Mantendo o Registro Mais Recente por Ingestão
df_info_dedup = deduplicar_por_ingestao(df_info_cleaned, "id_filme", "ingestion_datetime")

# Remoção da coluna de auditoria de ingestão antes da persistência na Silver
df_info_final = df_info_dedup.drop("ingestion_datetime")

# Persistência da Silver em Delta
(
    df_info_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_info_filmes")
)

print("Tabela silver.tb_info_filmes processada com SUCESSO!")

Tabela silver.tb_info_filmes processada com SUCESSO!


Processamento de Tabela: Financeiro dos Filmes

In [0]:
from pyspark.sql.types import DecimalType, StringType

df_financials_raw = spark.table(f"{catalogo}.{bronze_db}.tb_movies_financials")

# Pega a cotação mais recente disponível na tabela de cotação da API
df_cotacao_silver = spark.table(f"{catalogo}.{silver_db}.tb_cotacao_dolar")
taxa_cambio_atual = (
    df_cotacao_silver
    .orderBy(F.col("data_referencia").desc())
    .select("cotacao_dolar_compra")
    .first()[0]
)

print(f"Taxa de câmbio PTAX aplicada a partir da API: {taxa_cambio_atual}")

# Função para sanitizar e converter valores textuais de dinheiro
def tratar_dinheiro(col_name):
    raw_no_symbol = F.regexp_replace(F.trim(F.col(col_name)), r"[$\sR]", "")
    
    num_str = F.when(
        raw_no_symbol.contains(",") & raw_no_symbol.contains("."),
        F.when(
            F.instr(raw_no_symbol, ",") > F.instr(raw_no_symbol, "."),
            F.regexp_replace(F.regexp_replace(raw_no_symbol, r"\.", ""), ",", ".")
        ).otherwise(F.regexp_replace(raw_no_symbol, ",", ""))
    ).when(
        raw_no_symbol.contains(","), F.regexp_replace(raw_no_symbol, ",", ".")
    ).otherwise(raw_no_symbol)

    ext_val = F.regexp_extract(num_str, r"(-?[0-9]+(?:\.[0-9]+)?)", 1)
    num_clean = F.when(ext_val != "", ext_val).otherwise(F.lit(None)).cast("double")
    
    texto_upper = F.upper(raw_no_symbol)
    valor = (
        F.when(texto_upper.rlike(r"K\s*$"), num_clean * 1_000)
         .when(texto_upper.rlike(r"M\s*$"), num_clean * 1_000_000)
         .when(texto_upper.rlike(r"B\s*$"), num_clean * 1_000_000_000)
         .otherwise(num_clean)
    )
    
    return F.when(valor > 0, valor.cast(DecimalType(18, 2))).otherwise(F.lit(None))

# Limpeza e deduplicação da base financeira
df_fin_base = df_financials_raw.select(
    F.col("id").cast(StringType()).alias("id_filme"),
    tratar_dinheiro("budget").alias("orcamento_usd"),
    tratar_dinheiro("revenue").alias("receita_usd"),
    F.col("ingestion_datetime")
).filter(F.col("id_filme").isNotNull())

df_fin_dedup = deduplicar_por_ingestao(df_fin_base, "id_filme", "ingestion_datetime").drop("ingestion_datetime")

# Aplicação da cotação e cálculo de Lucro e Margem
df_financeiro_final = (
    df_fin_dedup
    .withColumn("orcamento_brl", F.round(F.col("orcamento_usd") * F.lit(taxa_cambio_atual), 2).cast(DecimalType(18, 2)))
    .withColumn("receita_brl", F.round(F.col("receita_usd") * F.lit(taxa_cambio_atual), 2).cast(DecimalType(18, 2)))
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast(DecimalType(18, 2)))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast(DecimalType(18, 2)))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("orcamento_usd").isNotNull() & (F.col("orcamento_usd") > 0), 
            F.round(((F.col("receita_usd") - F.col("orcamento_usd")) / F.col("orcamento_usd")) * 100, 2)
        ).otherwise(F.lit(None)).cast(DecimalType(18, 2))
    )
)

# Persistência na Silver em Delta
(
    df_financeiro_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_financeiro_filmes")
)

print("Tabela silver.tb_financeiro_filmes processada com SUCESSO!")

Taxa de câmbio PTAX aplicada a partir da API: 5.1111
Tabela silver.tb_financeiro_filmes processada com SUCESSO!


Processamento de Tabela: Métricas de Engajamento

In [0]:
df_metrics_raw = spark.table(f"{catalogo}.{bronze_db}.tb_movies_metrics")

# Limpeza de separadores decimais e conversão de Popularidade
pop_clean = F.regexp_replace(F.trim(F.col("popularity")), ",", ".")
pop_cast = (
    F.when(pop_clean.rlike(r"^[0-9]+(\.[0-9]+)?$"), pop_clean.cast(DoubleType()))
     .otherwise(F.lit(None))
)

# Validação: Garante que é >= 0
pop_valida = F.when(
    pop_cast.isNotNull() & 
    (pop_cast >= 0) & 
    (~pop_clean.rlike(r"^(19|20)\d{2}(\.0+)?$")), 
    pop_cast
).otherwise(F.lit(None))

# Tratamento para Nota TMDB
vote_avg_clean = F.regexp_replace(F.trim(F.col("vote_average")), ",", ".")
nota_tmdb_cast = (
    F.when(vote_avg_clean.rlike(r"^[0-9]+(\.[0-9]+)?$"), vote_avg_clean.cast(DoubleType()))
     .otherwise(F.lit(None))
)
nota_tmdb_valida = F.when((nota_tmdb_cast >= 0.0) & (nota_tmdb_cast <= 10.0), nota_tmdb_cast).otherwise(F.lit(None))

# Tratamento para Nota IMDb
avg_rating_clean = F.regexp_replace(F.trim(F.col("averageRating")), ",", ".")
nota_imdb_cast = (
    F.when(avg_rating_clean.rlike(r"^[0-9]+(\.[0-9]+)?$"), avg_rating_clean.cast(DoubleType()))
     .otherwise(F.lit(None))
)
nota_imdb_valida = F.when((nota_imdb_cast >= 0.0) & (nota_imdb_cast <= 10.0), nota_imdb_cast).otherwise(F.lit(None))

# Tratamento para Votos TMDB
vote_cnt_clean = F.trim(F.col("vote_count"))
votos_tmdb_cast = (
    F.when(vote_cnt_clean.rlike(r"^[0-9]+$"), vote_cnt_clean.cast(IntegerType()))
     .otherwise(F.lit(None))
)
votos_tmdb_validos = F.when(votos_tmdb_cast >= 0, votos_tmdb_cast).otherwise(F.lit(None))

# Tratamento para Votos IMDb
num_votes_clean = F.trim(F.col("numVotes"))
votos_imdb_cast = (
    F.when(num_votes_clean.rlike(r"^[0-9]+$"), num_votes_clean.cast(IntegerType()))
     .otherwise(F.lit(None))
)
votos_imdb_validos = F.when(votos_imdb_cast >= 0, votos_imdb_cast).otherwise(F.lit(None))

# Projeção, Mapeamento em Português e Tipagem
df_metrics_tratada = (
    df_metrics_raw
    .select(
        F.col("id").cast(StringType()).alias("id_filme"),
        pop_valida.alias("popularidade"),
        nota_tmdb_valida.alias("nota_media_tmdb"),
        votos_tmdb_validos.alias("qtd_votos_tmdb"),
        nota_imdb_valida.alias("nota_media_imdb"),
        votos_imdb_validos.alias("qtd_votos_imdb"),
        F.col("ingestion_datetime")
    )
    .filter(F.col("id_filme").isNotNull())
)

# Deduplicação Mantendo o Registro Mais Recente
df_metrics_dedup = deduplicar_por_ingestao(df_metrics_tratada, "id_filme", "ingestion_datetime")

# Remoção da coluna de controle de ingestão antes da gravação
df_metrics_final = df_metrics_dedup.drop("ingestion_datetime")

# Persistência na Silver em Delta
(
    df_metrics_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_metricas_engajamento")
)

print("Tabela silver.tb_metricas_engajamento processada com SUCESSO!")

Tabela silver.tb_metricas_engajamento processada com SUCESSO!


Processamento de Tabela: Avaliações de Usuários

In [0]:
df_reviews_raw = spark.table(f"{catalogo}.{bronze_db}.tb_movies_reviews")

# Conversão segura e validação da Nota do Usuário (Escala 0 a 10)
nota_cast = F.expr("try_cast(nota as double)")
nota_valida = F.when((nota_cast >= 0.0) & (nota_cast <= 10.0), nota_cast).otherwise(F.lit(None))

# Tratamento de Comentários Nulos, Vazios ou com Espaços em Branco
comentario_limpo = F.trim(F.col("comentario"))
comentario_tratado = F.when(
    F.col("comentario").isNull() | (comentario_limpo == ""), 
    F.lit("Sem comentário")
).otherwise(comentario_limpo)

# Projeção, Mapeamento em Português e Tipagem
df_reviews_tratada = (
    df_reviews_raw
    .select(
        F.col("id").cast(StringType()).alias("id_filme"),
        F.trim(F.col("nome")).cast(StringType()).alias("nome_usuario"),
        nota_valida.alias("nota_usuario"),
        comentario_tratado.alias("comentario_usuario")
    )
    .filter(F.col("id_filme").isNotNull())
)

# Deduplicação Integral
df_reviews_final = df_reviews_tratada.dropDuplicates([
    "id_filme", 
    "nome_usuario", 
    "nota_usuario", 
    "comentario_usuario"
])

# Persistência na Silver em Delta
(
    df_reviews_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_avaliacoes_usuarios")
)

print("Tabela silver.tb_avaliacoes_usuarios processada com SUCESSO!")

Tabela silver.tb_avaliacoes_usuarios processada com SUCESSO!


Processamento do Tabela: Catálogo de Gêneros

In [0]:
from pyspark.sql.types import StringType

df_credits_raw = spark.table(f"{catalogo}.{bronze_db}.tb_credits_and_tags")

# Lista dos 19 gêneros reconhecidos oficialmente (em minúsculas para padronização segura)
generos_validos_lower = [
    "action", "adventure", "animation", "comedy", "crime",
    "documentary", "drama", "family", "fantasy", "history",
    "horror", "music", "mystery", "romance", "science fiction",
    "tv movie", "thriller", "war", "western"
]

# Padronização de separadores, limpeza de ruídos e explode
df_generos_clean = (
    df_credits_raw
    .filter(F.col("id").isNotNull())
    .withColumn(
        "genero_raw",
        F.explode(F.split(F.regexp_replace(F.col("genres"), r"[;|]", ","), ","))
    )
    .select(
        F.col("id").cast(StringType()).alias("id_filme"),
        # Normaliza para minúsculas para ignorar problemas de maiúsculas/minúsculas na origem
        F.lower(F.trim(F.regexp_replace(F.col("genero_raw"), r"[-_]", " "))).alias("genero_normalizado"),
        F.initcap(F.trim(F.regexp_replace(F.col("genero_raw"), r"[-_]", " "))).alias("genero")
    )
    # Filtra usando a versão normalizada
    .filter(F.col("genero_normalizado").isin(generos_validos_lower))
    .select("id_filme", "genero")
    .dropDuplicates(["id_filme", "genero"])
)

# Persistência na Silver em Delta
(
    df_generos_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_generos")
)

print("Tabela silver.tb_generos processada com SUCESSO!")

Tabela silver.tb_generos processada com SUCESSO!


Processamento de Tabela: Pessoas e Empresas

In [0]:
df_credits_raw = spark.table(f"{catalogo}.{bronze_db}.tb_credits_and_tags")

# Função reutilizável com tratamento avançado e limpeza de Column Shift
def extrair_e_sanitizar_entidades(df_source, col_origem, rotulo_tipo_entidade):
    exploded = (
        df_source
        .filter(F.col("id").isNotNull() & F.col(col_origem).isNotNull())
        .select(
            F.col("id").cast(StringType()).alias("id_filme"),
            F.explode(
                F.split(F.regexp_replace(F.col(col_origem), r"[;|]", ","), ",")
            ).alias("entidade_raw")
        )
    )
    
    entidade_limpa = F.trim(F.regexp_replace(F.col("entidade_raw"), r"\s+", " "))
    
    return (
        exploded
        .select(
            F.col("id_filme"),
            entidade_limpa.alias("nome_entidade_raw"),
            F.lit(rotulo_tipo_entidade).alias("tipo_entidade")
        )
        .filter(
            F.col("nome_entidade_raw").isNotNull() & 
            (F.col("nome_entidade_raw") != "") &
            (~F.upper(F.col("nome_entidade_raw")).isin("[]", "N/A", "UNKNOWN", "NÃO INFORMADO", "NULL", "NONE")) &
            (~F.lower(F.col("nome_entidade_raw")).rlike(r"^/|.*\.(jpg|jpeg|png|webp)$")) &
            (~F.col("nome_entidade_raw").rlike(r"^\d+(\.\d+)?$")) &
            (F.length(F.col("nome_entidade_raw")) <= 100)
        )
        .select(
            F.col("id_filme"),
            F.initcap(F.col("nome_entidade_raw")).alias("nome_entidade"),
            F.col("tipo_entidade")
        )
    )

# Extração dos 4 fluxos categorizados
df_atores = extrair_e_sanitizar_entidades(df_credits_raw, "cast", "Ator")
df_diretores = extrair_e_sanitizar_entidades(df_credits_raw, "directors", "Diretor")
df_roteiristas = extrair_e_sanitizar_entidades(df_credits_raw, "writers", "Roteirista")
df_produtoras = extrair_e_sanitizar_entidades(df_credits_raw, "production_companies", "Produtora")

# Unificação dos DataFrames e Deduplicação Integral
df_pessoas_empresas_final = (
    df_atores
    .union(df_diretores)
    .union(df_roteiristas)
    .union(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

# Persistência na Silver em Delta
(
    df_pessoas_empresas_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.{silver_db}.tb_pessoas_empresas")
)

print("Tabela silver.tb_pessoas_empresas processada com SUCESSO!")

Tabela silver.tb_pessoas_empresas processada com SUCESSO!


Otimização e Clusterização das Tabelas Silver

In [0]:
tabelas_por_filme = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas"
]

# Execução da compactação de pequenos arquivos e Z-Ordering por id_filme
# Habilita Data Skipping no Delta Lake para acelerar os JOINs da Gold
print("Iniciando otimização física das tabelas Silver...")

for tabela in tabelas_por_filme:
    tabela_full = f"{catalogo}.{silver_db}.{tabela}"
    print(f"Otimizando {tabela_full} (ZORDER BY id_filme)...")
    spark.sql(f"OPTIMIZE {tabela_full} ZORDER BY (id_filme)")

# Execução da compactação e Z-Ordering
tabela_cotacao = f"{catalogo}.{silver_db}.tb_cotacao_dolar"
print(f"Otimizando {tabela_cotacao} (ZORDER BY data_referencia)...")
spark.sql(f"OPTIMIZE {tabela_cotacao} ZORDER BY (data_referencia)")

print("\nOtimização física (OPTIMIZE + ZORDER BY) concluída com SUCESSO na Camada Silver!")

Iniciando otimização física das tabelas Silver...
Otimizando medalhao.silver.tb_info_filmes (ZORDER BY id_filme)...
Otimizando medalhao.silver.tb_financeiro_filmes (ZORDER BY id_filme)...
Otimizando medalhao.silver.tb_metricas_engajamento (ZORDER BY id_filme)...
Otimizando medalhao.silver.tb_avaliacoes_usuarios (ZORDER BY id_filme)...
Otimizando medalhao.silver.tb_generos (ZORDER BY id_filme)...
Otimizando medalhao.silver.tb_pessoas_empresas (ZORDER BY id_filme)...
Otimizando medalhao.silver.tb_cotacao_dolar (ZORDER BY data_referencia)...

Otimização física (OPTIMIZE + ZORDER BY) concluída com SUCESSO na Camada Silver!
